# StockMind AI: Cloud Data Audit & Environment Diagnostic

**Project:** StockMind AI: A Big Data-Driven Generative AI Framework for Indian Stock Market Analysis and Forecasting  
**Academic Coursework:** Generative AI & Big Data Analytics  
**Purpose:** Reproducible Google Colab diagnostic and data auditing notebook designed to run memory-safely on standard CPU runtimes or GPU-accelerated environments.

---

### Step 1: System Hardware, RAM & GPU Diagnostic
Checks available host resources (RAM, CPU cores, GPU if attached, disk space) and verifies standard dependencies without assuming a GPU is present.

In [ ]:
import os
import sys
import platform
import shutil
import psutil

print("=" * 60)
print(" STOCKMIND AI - SYSTEM HARDWARE DIAGNOSTIC")
print("=" * 60)
print(f"OS Platform        : {platform.system()} {platform.release()} ({platform.machine()})")
print(f"Python Version     : {platform.python_version()}")
print(f"Logical CPU Cores  : {os.cpu_count()}")

# RAM check
mem = psutil.virtual_memory()
print(f"Total RAM          : {mem.total / (1024**3):.2f} GB")
print(f"Available RAM      : {mem.available / (1024**3):.2f} GB ({mem.percent}% used)")

# Disk check
disk = shutil.disk_usage(".")
print(f"Disk Space Total   : {disk.total / (1024**3):.2f} GB")
print(f"Disk Space Free    : {disk.free / (1024**3):.2f} GB")

# GPU check (Safe check, works gracefully if no GPU exists)
gpu_detected = False
try:
    import torch
    if torch.cuda.is_available():
        gpu_detected = True
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU Detected       : {gpu_name} ({gpu_mem:.2f} GB VRAM)")
    else:
        print("GPU Detected       : None (Running on standard CPU runtime)")
except ImportError:
    print("GPU Detected       : Torch not installed; skipping CUDA probe")

print("=" * 60)

### Step 2: Environment Setup & Optional User-Approved Google Drive Mount
Configure input/output directories. If running on Google Colab, you can mount Google Drive with explicit user authorization.

In [ ]:
# Set configuration paths
MOUNT_GOOGLE_DRIVE = False   # Set to True in Google Colab if datasets are stored in Google Drive
DRIVE_MOUNT_POINT = "/content/drive"

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        print("[*] Requesting user authorization to mount Google Drive...")
        drive.mount(DRIVE_MOUNT_POINT)
        print("[+] Google Drive mounted successfully at:", DRIVE_MOUNT_POINT)
        BASE_DIR = os.path.join(DRIVE_MOUNT_POINT, "MyDrive/stockmind")
    except ImportError:
        print("[!] Note: Not running inside Google Colab. Using local repository path.")
        BASE_DIR = "."
else:
    BASE_DIR = "."

DATASET_DIR = os.path.join(BASE_DIR, "Dataset")
OUTPUT_DIR = os.path.join(BASE_DIR, "artifacts/colab_audit")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Dataset Directory  : {os.path.abspath(DATASET_DIR)}")
print(f"Artifact Output Dir: {os.path.abspath(OUTPUT_DIR)}")
print(f"Dataset Available  : {os.path.exists(DATASET_DIR)}")

### Step 3: Install Required Lightweight Dependencies
Installs `pyarrow`, `pandas`, and `openpyxl` for out-of-core metadata inspection.

In [ ]:
!pip install --quiet pyarrow pandas openpyxl

### Step 4: Run Safe Streaming Data Audit
Uses zero-copy Parquet metadata and streaming CSV line counting to inspect file sizes, row counts, and date bounds without loading large files into RAM.

In [ ]:
import glob
import json
import pandas as pd
import pyarrow.parquet as pq

audit_results = {
    "system": {
        "platform": platform.platform(),
        "ram_gb": round(mem.total / (1024**3), 2),
        "gpu_available": gpu_detected,
    },
    "categories": {},
}

# 1. Audit NSE Daily Parquet files (if available)
parquet_pattern = os.path.join(DATASET_DIR, "market_data/nse_daily/*.parquet")
parquet_files = sorted(glob.glob(parquet_pattern))
print(f"[*] Found {len(parquet_files)} NSE Daily Parquet files")

nse_summary = {}
total_rows = 0
for pf_path in parquet_files:
    pf = pq.ParquetFile(pf_path)
    fname = os.path.basename(pf_path)
    rows = pf.metadata.num_rows
    cols = pf.metadata.num_columns
    size_mb = os.path.getsize(pf_path) / (1024**2)
    total_rows += rows
    nse_summary[fname] = {"rows": rows, "cols": cols, "size_mb": round(size_mb, 2)}
    print(f"    - {fname}: {rows:,} rows, {cols} cols, {size_mb:.2f} MB")

audit_results["categories"]["nse_daily"] = {
    "file_count": len(parquet_files),
    "total_rows": total_rows,
    "files": nse_summary,
}

# 2. Audit Bulk Deals (Corporate Events)
bulk_csv = os.path.join(DATASET_DIR, "corporate_events/Bulk-Deals-25-10-2025-to-25-10-2026.csv")
if os.path.exists(bulk_csv):
    bulk_df = pd.read_csv(bulk_csv, nrows=500)
    print(f"\n[*] Inspected Bulk Deals sample (500 rows):")
    print(f"    Columns: {list(bulk_df.columns)}")
    audit_results["categories"]["bulk_deals"] = {
        "sample_rows": len(bulk_df),
        "raw_columns": list(bulk_df.columns),
        "whitespace_headers": [c for c in bulk_df.columns if c != c.strip()],
    }

# 3. Audit Financial News (Sample)
news_csv = os.path.join(DATASET_DIR, "financial_news/training_data_26000.csv")
if os.path.exists(news_csv):
    news_df = pd.read_csv(news_csv, nrows=100)
    print(f"\n[*] Inspected Financial News sample (100 rows):")
    print(f"    Columns: {list(news_df.columns)}")
    print(f"    Sentiments: {news_df['Sentiment'].value_counts().to_dict()}")
    audit_results["categories"]["financial_news"] = {
        "sample_rows": len(news_df),
        "columns": list(news_df.columns),
        "sentiment_sample": news_df["Sentiment"].value_counts().to_dict(),
    }

# Save audit output to persistent directory
out_report_path = os.path.join(OUTPUT_DIR, "colab_audit_report.json")
with open(out_report_path, "w", encoding="utf-8") as f:
    json.dump(audit_results, f, indent=2)
print(f"\n[+] Saved persistent audit report to: {out_report_path}")

### Step 5: Summary & Artifact Export
Displays the final validation status and provides instructions on copying the generated artifacts.

In [ ]:
print("=" * 60)
print(" AUDIT JOB COMPLETED SUCCESSFULLY")
print("=" * 60)
print(f"Persistent artifact path: {os.path.abspath(out_report_path)}")
print("To sync back to your local repository:")
print("1. Download colab_audit_report.json from the Colab file browser.")
print("2. Place it in Desktop/stockmind/artifacts/colab_audit_report.json.")
print("3. Antigravity will consume the verified report locally.")
print("=" * 60)